# NSE 5× Turnaround Breakout Research — DuckDB

Fixed/improved research notebook based on the previous 5× run.

**Main fixes:** price-quality anomaly protection, minimum price/liquidity filters, de-duplicated winner episodes, next-open execution, MAE/MFE, realistic exits, chronological train/validation/test evaluation, and compact exports.

## 1. Configuration

In [ ]:
from pathlib import Path
import json, warnings
warnings.filterwarnings("ignore")

USE_COLAB_DRIVE=True
DATA_DIR=Path("/content/drive/MyDrive/quant/data/parquet/daily")
OUTPUT_DIR=Path("/content/drive/MyDrive/quant/results/5x_turnaround_breakout_v2")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
PARQUET_GLOB=str(DATA_DIR/"**"/"*.parquet")
DB_PATH=str(OUTPUT_DIR/"research.duckdb")

MIN_PRICE=20.0
MIN_MEDIAN_TURNOVER_60D=5_000_000.0
MIN_HISTORY=252
FORWARD_DAYS=252
WINNER_MULTIPLE=5.0

ENABLE_PRICE_JUMP_SCREEN=True
REQUIRE_CLEAN_SYMBOL=True
PRICE_JUMP_UPPER=5.0
PRICE_JUMP_LOWER=0.20
MIN_VALID_PRICE=0.01
MAX_VALID_PRICE=1_000_000.0

BREAKOUT_DAYS=60
LOWER_RANGE_POSITION_MAX=0.40
MIN_DRAWDOWN_FROM_252_HIGH=0.30
MAX_DRAWDOWN_FROM_252_HIGH=0.80
BREAKOUT_VOLUME_MULTIPLE=1.50
MIN_ENTRY_SCORE=7

INITIAL_STOP=0.30
FAILURE_DAYS=252
ENABLE_TRAILING_STOP=True
TRAILING_STOP=0.25
TRAIL_START_RETURN=1.00
SIGNAL_COOLDOWN_DAYS=60

TRAIN_END="2018-12-31"
VALIDATION_END="2021-12-31"
TEST_END="2024-12-31"

In [ ]:
if USE_COLAB_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive",force_remount=False)
    except Exception as e:
        print("Drive:",e)

!pip -q install -U duckdb pyarrow pandas numpy matplotlib

import duckdb, pandas as pd, numpy as np, matplotlib.pyplot as plt
con=duckdb.connect(DB_PATH)
con.execute("PRAGMA threads=4")
con.execute("PRAGMA enable_progress_bar=false")
print("DuckDB",duckdb.__version__)

## 2. Discover Parquet schema

In [ ]:
files=list(DATA_DIR.rglob("*.parquet"))
if not files: raise FileNotFoundError(f"No Parquet files under {DATA_DIR}")
sample=str(files[0]).replace("'","''")
schema=con.execute("DESCRIBE SELECT * FROM read_parquet('"+sample+"')").df()
display(schema)

def col(candidates):
    m={x.lower():x for x in schema.column_name.tolist()}
    for c in candidates:
        if c.lower() in m: return m[c.lower()]
    raise ValueError(f"Missing {candidates}; columns={list(m.values())}")

DATE_COL=col(["date","datetime","timestamp","trade_date"])
SYMBOL_COL=col(["symbol","ticker","tradingsymbol","security"])
OPEN_COL=col(["open","open_price"])
HIGH_COL=col(["high","high_price"])
LOW_COL=col(["low","low_price"])
CLOSE_COL=col(["close","close_price","last_price","ltp"])
VOLUME_COL=col(["volume","total_traded_quantity","qty","quantity"])
print(DATE_COL,SYMBOL_COL,OPEN_COL,HIGH_COL,LOW_COL,CLOSE_COL,VOLUME_COL)

## 3. Normalized DuckDB views

In [ ]:
def qi(x): return '"'+str(x).replace('"','""')+'"'
glob_sql=PARQUET_GLOB.replace("'","''")

con.execute("DROP VIEW IF EXISTS daily_raw")
con.execute("DROP VIEW IF EXISTS daily")

sql = (
    "CREATE VIEW daily_raw AS SELECT "
    "CAST("+qi(DATE_COL)+" AS DATE) AS date, "
    "TRIM(CAST("+qi(SYMBOL_COL)+" AS VARCHAR)) AS symbol, "
    "TRY_CAST("+qi(OPEN_COL)+" AS DOUBLE) AS open, "
    "TRY_CAST("+qi(HIGH_COL)+" AS DOUBLE) AS high, "
    "TRY_CAST("+qi(LOW_COL)+" AS DOUBLE) AS low, "
    "TRY_CAST("+qi(CLOSE_COL)+" AS DOUBLE) AS close, "
    "TRY_CAST("+qi(VOLUME_COL)+" AS DOUBLE) AS volume "
    "FROM read_parquet('"+glob_sql+"',union_by_name=true,hive_partitioning=true)"
)
con.execute(sql)

con.execute("""
CREATE VIEW daily AS
WITH x AS (
 SELECT *,ROW_NUMBER() OVER(PARTITION BY symbol,date ORDER BY date) AS rn
 FROM daily_raw
)
SELECT date,symbol,open,high,low,close,volume FROM x WHERE rn=1
""")

display(con.execute("""
SELECT COUNT(*) AS total_rows,
COUNT(DISTINCT symbol) AS total_symbols,
MIN(date) AS first_date,MAX(date) AS last_date
FROM daily
""").df())

## 4. Data-quality / price-scale anomaly screen

In [ ]:
con.execute("DROP VIEW IF EXISTS quality_base")
con.execute("""
CREATE VIEW quality_base AS
WITH x AS (
 SELECT *,LAG(close) OVER(PARTITION BY symbol ORDER BY date) AS prev_close
 FROM daily
)
SELECT *,
 CASE
  WHEN close IS NULL OR close<=0 OR open IS NULL OR high IS NULL OR low IS NULL THEN TRUE
  WHEN high<low OR high<open OR high<close OR low>open OR low>close THEN TRUE
  WHEN close<0.01 OR close>1_000_000.0 THEN TRUE
  ELSE FALSE
 END AS bad_ohlc,
 CASE
  WHEN prev_close IS NULL OR prev_close<=0 OR close IS NULL THEN FALSE
  WHEN close/prev_close>5.0 OR close/prev_close<0.20 THEN TRUE
  ELSE FALSE
 END AS price_jump_flag
FROM x
""")

quality=con.execute("""
SELECT COUNT(*) AS total_rows,
SUM(CASE WHEN bad_ohlc THEN 1 ELSE 0 END) AS bad_ohlc_rows,
SUM(CASE WHEN price_jump_flag THEN 1 ELSE 0 END) AS jump_rows,
COUNT(DISTINCT CASE WHEN price_jump_flag THEN symbol END) AS symbols_with_jumps
FROM quality_base
""").df()
display(quality)

jumps=con.execute("""
SELECT symbol,COUNT(*) FILTER(WHERE price_jump_flag) AS jump_count,
MIN(date) FILTER(WHERE price_jump_flag) AS first_jump_date,
MAX(date) FILTER(WHERE price_jump_flag) AS last_jump_date
FROM quality_base GROUP BY symbol
HAVING COUNT(*) FILTER(WHERE price_jump_flag)>0
ORDER BY jump_count DESC LIMIT 100
""").df()
display(jumps.head(50))

**Why this matters:** the previous run contained extreme examples such as a stock moving from ₹9.60 to ₹7,680. A raw future-price label can turn a price-scale error into a fake 800× winner. The default clean universe excludes symbols with extreme discontinuities.

In [ ]:
con.execute("DROP VIEW IF EXISTS symbol_quality")
con.execute("""
CREATE VIEW symbol_quality AS
SELECT symbol,COUNT(*) AS rows_count,MIN(date) AS first_date,MAX(date) AS last_date,
COUNT(*) FILTER(WHERE bad_ohlc) AS bad_ohlc_rows,
COUNT(*) FILTER(WHERE price_jump_flag) AS price_jump_rows
FROM quality_base GROUP BY symbol
""")
con.execute("DROP VIEW IF EXISTS clean_daily")
if REQUIRE_CLEAN_SYMBOL and ENABLE_PRICE_JUMP_SCREEN:
    con.execute("""
    CREATE VIEW clean_daily AS
    SELECT q.date,q.symbol,q.open,q.high,q.low,q.close,q.volume
    FROM quality_base q JOIN symbol_quality s USING(symbol)
    WHERE NOT q.bad_ohlc AND s.price_jump_rows=0
    """)
else:
    con.execute("""
    CREATE VIEW clean_daily AS
    SELECT date,symbol,open,high,low,close,volume
    FROM quality_base WHERE NOT bad_ohlc
    """)
display(con.execute("""
SELECT COUNT(*) AS clean_rows,COUNT(DISTINCT symbol) AS clean_symbols,
MIN(date) AS first_date,MAX(date) AS last_date FROM clean_daily
""").df())

## 5. DuckDB feature engine

In [ ]:
con.execute("DROP VIEW IF EXISTS features")
con.execute("""
CREATE VIEW features AS
WITH b AS (
 SELECT *,
 ROW_NUMBER() OVER(PARTITION BY symbol ORDER BY date) AS obs_num,
 LAG(close) OVER(PARTITION BY symbol ORDER BY date) AS prev_close,
 LAG(close,20) OVER(PARTITION BY symbol ORDER BY date) AS c20,
 LAG(close,60) OVER(PARTITION BY symbol ORDER BY date) AS c60,
 LAG(close,120) OVER(PARTITION BY symbol ORDER BY date) AS c120,
 LAG(close,252) OVER(PARTITION BY symbol ORDER BY date) AS c252,
 LAG(low,20) OVER(PARTITION BY symbol ORDER BY date) AS low20ago,
 LAG(high,20) OVER(PARTITION BY symbol ORDER BY date) AS high20ago,
 GREATEST(high-low,ABS(high-prev_close),ABS(low-prev_close)) AS tr
 FROM clean_daily
),
r AS (
 SELECT *,
 AVG(close) OVER(PARTITION BY symbol ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS sma20,
 AVG(close) OVER(PARTITION BY symbol ORDER BY date ROWS BETWEEN 49 PRECEDING AND CURRENT ROW) AS sma50,
 AVG(close) OVER(PARTITION BY symbol ORDER BY date ROWS BETWEEN 99 PRECEDING AND CURRENT ROW) AS sma100,
 AVG(close) OVER(PARTITION BY symbol ORDER BY date ROWS BETWEEN 199 PRECEDING AND CURRENT ROW) AS sma200,
 AVG(tr) OVER(PARTITION BY symbol ORDER BY date ROWS BETWEEN 13 PRECEDING AND CURRENT ROW) AS atr14,
 STDDEV_SAMP(LN(close/NULLIF(prev_close,0))) OVER(PARTITION BY symbol ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS vol20,
 STDDEV_SAMP(LN(close/NULLIF(prev_close,0))) OVER(PARTITION BY symbol ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW) AS vol60,
 MAX(close) OVER(PARTITION BY symbol ORDER BY date ROWS BETWEEN 60 PRECEDING AND 1 PRECEDING) AS hi60,
 MAX(close) OVER(PARTITION BY symbol ORDER BY date ROWS BETWEEN 252 PRECEDING AND 1 PRECEDING) AS hi252,
 MIN(close) OVER(PARTITION BY symbol ORDER BY date ROWS BETWEEN 252 PRECEDING AND 1 PRECEDING) AS lo252,
 AVG(volume) OVER(PARTITION BY symbol ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS vol20avg,
 AVG(volume) OVER(PARTITION BY symbol ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW) AS vol60avg,
 MEDIAN(close*volume) OVER(PARTITION BY symbol ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW) AS turnover60
 FROM b
),
s AS (
 SELECT *,
 LAG(sma20,20) OVER(PARTITION BY symbol ORDER BY date) AS sma20ago
 FROM r
)
SELECT *,
 close/NULLIF(c20,0)-1 AS ret20,
 close/NULLIF(c60,0)-1 AS ret60,
 close/NULLIF(c120,0)-1 AS ret120,
 close/NULLIF(c252,0)-1 AS ret252,
 close/NULLIF(hi252,0)-1 AS dd252,
 (close-lo252)/NULLIF(hi252-lo252,0) AS range_pos252,
 close/NULLIF(sma50,0)-1 AS close_vs_sma50,
 close/NULLIF(sma20,0)-1 AS close_vs_sma20,
 sma20/NULLIF(sma20ago,0)-1 AS sma20_slope,
 volume/NULLIF(vol20avg,0) AS volume_ratio20,
 atr14/NULLIF(close,0) AS atr_pct,
 close>hi60 AS breakout60,
 close>sma20 AS above_sma20,
 sma20>sma20ago AS sma20_rising,
 low>low20ago AS higher_low,
 high>high20ago AS higher_high
FROM s
""")
display(con.execute("""
SELECT COUNT(*) AS rows,COUNT(DISTINCT symbol) AS symbols,
MIN(date) AS first_date,MAX(date) AS last_date FROM features WHERE obs_num>=?
""",[MIN_HISTORY]).df())

## 6. Historical 5× labels — deduplicated

In [ ]:
con.execute("DROP VIEW IF EXISTS outcome")
con.execute("""
CREATE VIEW outcome AS
SELECT f.*,
COUNT(*) OVER(PARTITION BY symbol ORDER BY date ROWS BETWEEN 1 FOLLOWING AND 252 FOLLOWING) AS future_n,
MAX(close) OVER(PARTITION BY symbol ORDER BY date ROWS BETWEEN 1 FOLLOWING AND 252 FOLLOWING) AS future_max
FROM features f WHERE obs_num>=252
""")

con.execute("DROP VIEW IF EXISTS five_x_events")
con.execute("""
CREATE VIEW five_x_events AS
WITH w AS (
 SELECT *,future_max/NULLIF(close,0)-1 AS future_max_return,
        LAG(date) OVER(PARTITION BY symbol ORDER BY date) AS prev_winner
 FROM outcome
 WHERE future_n=252 AND future_max>=close*5.0
)
SELECT * FROM w
WHERE prev_winner IS NULL OR date_diff('day',prev_winner,date)>60
""")

display(con.execute("""
SELECT COUNT(*) AS events,COUNT(DISTINCT symbol) AS symbols,
MIN(date) AS first_event,MAX(date) AS last_event FROM five_x_events
""").df())

## 7. Turnaround-breakout candidate score

In [ ]:
con.execute("DROP VIEW IF EXISTS candidate_signals")
con.execute("""
CREATE VIEW candidate_signals AS
WITH x AS (
 SELECT f.*,
  (CASE WHEN dd252<=-0.30 AND dd252>=-0.80 THEN 1 ELSE 0 END
  +CASE WHEN range_pos252<=0.40 THEN 1 ELSE 0 END
  +CASE WHEN vol20>0 THEN 1 ELSE 0 END
  +CASE WHEN above_sma20 THEN 1 ELSE 0 END
  +CASE WHEN sma20_rising THEN 1 ELSE 0 END
  +CASE WHEN higher_low THEN 1 ELSE 0 END
  +CASE WHEN higher_high THEN 1 ELSE 0 END
  +CASE WHEN breakout60 THEN 1 ELSE 0 END
  +CASE WHEN volume_ratio20>=1.50 THEN 1 ELSE 0 END
  +CASE WHEN ret20>0 THEN 1 ELSE 0 END) AS score
 FROM features f
 WHERE obs_num>=252 AND close>=20.0 AND turnover60>=5_000_000.0
)
SELECT *,CASE WHEN score>=7 AND breakout60 AND volume_ratio20>=1.50 THEN TRUE ELSE FALSE END AS entry_signal
FROM x
""")
display(con.execute("""
SELECT COUNT(*) FILTER(WHERE entry_signal) AS signals,
COUNT(DISTINCT symbol) FILTER(WHERE entry_signal) AS symbols,
MIN(date) FILTER(WHERE entry_signal) AS first_signal,
MAX(date) FILTER(WHERE entry_signal) AS last_signal
FROM candidate_signals
""").df())

### Signal design

The system does **not** buy a falling knife. It requires a depressed setup plus evidence of stabilization and an actual breakout with volume. The 7/10 score is a starting research parameter, not an optimized result.

In [ ]:
con.execute("DROP VIEW IF EXISTS signal_events")
con.execute("""
CREATE VIEW signal_events AS
WITH s AS (
 SELECT symbol,date AS signal_date,close AS signal_close,score,
        turnover60,dd252,range_pos252,ret20,ret60,volume_ratio20,
        sma20_rising,higher_low,higher_high,breakout60
 FROM candidate_signals WHERE entry_signal
),
p AS (
 SELECT *,LAG(signal_date) OVER(PARTITION BY symbol ORDER BY signal_date) AS prev_signal
 FROM s
)
SELECT * FROM p
WHERE prev_signal IS NULL OR date_diff('day',prev_signal,signal_date)>=60
""")
display(con.execute("""
SELECT COUNT(*) AS signals,COUNT(DISTINCT symbol) AS symbols,
AVG(score) AS avg_score,MIN(signal_date) AS first_signal,MAX(signal_date) AS last_signal
FROM signal_events
""").df())

## 8. Next-session-open entries

In [ ]:
con.execute("DROP VIEW IF EXISTS entries")
con.execute("""
CREATE VIEW entries AS
SELECT s.*,d.date AS entry_date,d.open AS entry_price
FROM signal_events s JOIN clean_daily d ON d.symbol=s.symbol
AND d.date=(SELECT MIN(d2.date) FROM clean_daily d2
           WHERE d2.symbol=s.symbol AND d2.date>s.signal_date)
WHERE d.open IS NOT NULL AND d.open>0
""")
display(con.execute("""
SELECT COUNT(*) AS trades,COUNT(DISTINCT symbol) AS symbols,
MIN(entry_date) AS first_entry,MAX(entry_date) AS last_entry FROM entries
""").df())

## 9. MAE/MFE paths

In [ ]:
paths=con.execute("""
SELECT e.symbol,e.signal_date,e.entry_date,e.entry_price,e.score,
       d.date,d.open,d.high,d.low,d.close,
       ROW_NUMBER() OVER(PARTITION BY e.symbol,e.entry_date ORDER BY d.date)-1 AS day
FROM entries e JOIN clean_daily d ON d.symbol=e.symbol AND d.date>=e.entry_date
""").df()
paths=paths[paths.day<FAILURE_DAYS].copy()
print("Path rows:",len(paths))

## 10. Executable exit model

In [ ]:
def simulate(g):
    g=g.sort_values("day").reset_index(drop=True)
    entry=float(g.entry_price.iloc[0])
    hi=entry; lo=entry; highest_close=entry
    trailing=False; reason="TIME"; exit_day=int(g.day.iloc[-1]); exit_price=float(g.close.iloc[-1])
    for _,r in g.iterrows():
        day=int(r.day); high=float(r.high); low=float(r.low); close=float(r.close)
        hi=max(hi,high); lo=min(lo,low); highest_close=max(highest_close,close)
        stop=entry*(1-INITIAL_STOP)
        if low<=stop:
            reason,exit_day,exit_price="INITIAL_STOP",day,stop; break
        if close>=entry*(1+TRAIL_START_RETURN): trailing=ENABLE_TRAILING_STOP
        if trailing:
            trail=highest_close*(1-TRAILING_STOP)
            if low<=trail:
                reason,exit_day,exit_price="TRAILING_STOP",day,trail; break
        if day==FAILURE_DAYS-1:
            reason,exit_day,exit_price="TIME",day,close; break
    return pd.Series({
        "symbol":g.symbol.iloc[0],"signal_date":g.signal_date.iloc[0],
        "entry_date":g.entry_date.iloc[0],"entry_price":entry,"score":g.score.iloc[0],
        "exit_day":exit_day,"exit_price":exit_price,"exit_reason":reason,
        "trade_return":exit_price/entry-1,"mfe":hi/entry-1,"mae":lo/entry-1,
        "hit_2x":hi>=entry*2,"hit_3x":hi>=entry*3,"hit_5x":hi>=entry*6
    })

trades=(paths.groupby(["symbol","entry_date"],group_keys=False).apply(simulate).reset_index(drop=True)
        if len(paths) else pd.DataFrame())
if not trades.empty: trades.entry_date=pd.to_datetime(trades.entry_date)
print("Trades:",len(trades))
display(trades.head())

## 11. Performance, MAE/MFE and walk-forward

In [ ]:
def summ(df,label):
    if df.empty: return {"period":label,"trades":0}
    return {"period":label,"trades":len(df),"win_rate":(df.trade_return>0).mean(),
            "median_return":df.trade_return.median(),"mean_return":df.trade_return.mean(),
            "hit_2x":df.hit_2x.mean(),"hit_3x":df.hit_3x.mean(),"hit_5x":df.hit_5x.mean(),
            "median_exit_day":df.exit_day.median(),"median_mae":df.mae.median(),"median_mfe":df.mfe.median()}

if not trades.empty:
    train=trades[trades.entry_date<=pd.Timestamp(TRAIN_END)]
    validation=trades[(trades.entry_date>pd.Timestamp(TRAIN_END))&(trades.entry_date<=pd.Timestamp(VALIDATION_END))]
    test=trades[(trades.entry_date>pd.Timestamp(VALIDATION_END))&(trades.entry_date<=pd.Timestamp(TEST_END))]
    summary=pd.DataFrame([summ(train,"TRAIN"),summ(validation,"VALIDATION"),summ(test,"TEST"),summ(trades,"ALL")])
    display(summary)
    display(trades.groupby("exit_reason").agg(trades=("trade_return","size"),
        median_return=("trade_return","median"),mean_return=("trade_return","mean")))
    mae_bins=pd.cut(trades.mae,[-1,-.75,-.50,-.40,-.30,-.20,-.10,0],include_lowest=True)
    display(trades.groupby(mae_bins,observed=False).agg(trades=("trade_return","size"),
        median_return=("trade_return","median"),hit_2x=("hit_2x","mean"),hit_3x=("hit_3x","mean"),hit_5x=("hit_5x","mean")))
    display(trades.groupby("score").agg(trades=("trade_return","size"),
        median_return=("trade_return","median"),hit_2x=("hit_2x","mean"),hit_5x=("hit_5x","mean")))
else:
    print("No trades.")

## 12. Monte Carlo sequence risk

In [ ]:
def mc(df,n=2000,risk=.02,capital=100000):
    if df.empty:return pd.DataFrame()
    x=df.trade_return.clip(-.99,None).to_numpy(); out=[]
    for _ in range(n):
        seq=np.random.choice(x,len(x),replace=True); eq=capital; peak=eq; dd=0
        for r in seq:
            eq*=1+risk*r; peak=max(peak,eq); dd=min(dd,eq/peak-1)
        out.append((eq,eq/capital-1,dd))
    return pd.DataFrame(out,columns=["final_equity","return","max_drawdown"])
if not trades.empty and not test.empty:
    mc_df=mc(test); display(mc_df.describe(percentiles=[.01,.05,.5,.95,.99]))
    plt.figure(figsize=(10,5)); plt.hist(mc_df.max_drawdown,bins=50)
    plt.title("Monte Carlo Maximum Drawdown — Test"); plt.xlabel("Drawdown"); plt.ylabel("Paths"); plt.show()

## 13. Export

In [ ]:
candidate_export=con.execute("""
SELECT symbol,date,close,score,entry_signal,dd252,range_pos252,ret20,ret60,ret120,ret252,
       vol20,vol60,volume_ratio20,close_vs_sma50,sma20_rising,higher_low,higher_high,
       breakout60,turnover60
FROM candidate_signals WHERE entry_signal ORDER BY date,symbol
""").df()
candidate_export.to_parquet(OUTPUT_DIR/"candidate_entry_signals.parquet",index=False)
candidate_export.to_csv(OUTPUT_DIR/"candidate_entry_signals.csv",index=False)

winner_export=con.execute("""
SELECT symbol,date,close,future_max,future_max_return,dd252,range_pos252,
       ret20,ret60,ret120,ret252,vol20,vol60,volume_ratio20,turnover60
FROM five_x_events ORDER BY date,symbol
""").df()
winner_export.to_parquet(OUTPUT_DIR/"deduped_historical_5x_events.parquet",index=False)
winner_export.to_csv(OUTPUT_DIR/"deduped_historical_5x_events.csv",index=False)

if not trades.empty:
    trades.to_parquet(OUTPUT_DIR/"backtest_trades.parquet",index=False)
    trades.to_csv(OUTPUT_DIR/"backtest_trades.csv",index=False)

print("Output:",OUTPUT_DIR)
print("Candidate signals:",len(candidate_export))
print("Historical 5x episodes:",len(winner_export))
print("Trades:",len(trades))

# Interpretation

The historical scan is a **discovery study**; the executable backtest is the important test.

Before live use, add:
- verified corporate-action-adjusted data,
- survivorship-bias controls,
- brokerage/slippage,
- portfolio position sizing and concurrent positions,
- sector exposure limits,
- benchmark-relative momentum,
- rolling walk-forward optimization.

Do not tune the test period. Use TRAIN → VALIDATION → freeze → TEST.